# 10 - Fairness and compliance testing

Tests whether the PD scorecard treats groups differently, even though gender is not used in the model. Credit scoring of individuals is a high-risk AI use case under the EU AI Act, and excluding a sensitive variable does not guarantee fair outcomes: other variables can act as proxies.

Gender is joined back from the raw data by SK_ID_CURR **for testing only**; it is never used in the model. Age groups are also tested, since age is a sensitive characteristic. The analysis uses the test set.

### Scores and sensitive attributes

By gender and age

In [1]:
import sys
sys.path.append("..")

import joblib
import numpy as np
import pandas as pd
from src.woe import load_binners, woe_transform

test = pd.read_parquet("../data/processed/test.parquet")
binners = load_binners("../models/binners.joblib")
saved = joblib.load("../models/pd_model.joblib")
model, FINAL_VARS = saved["model"], saved["final_vars"]

X_test = woe_transform(test, binners)[FINAL_VARS]
test["PD"] = model.predict_proba(X_test)[:, 1]
test["SCORE"] = saved["offset"] - saved["factor"] * model.decision_function(X_test)

# Join gender back from the raw data, for testing only
raw = pd.read_csv("../data/raw/application_train.csv", usecols=["SK_ID_CURR", "CODE_GENDER"])
test = test.merge(raw, on="SK_ID_CURR", how="left")
test = test[test["CODE_GENDER"].isin(["F", "M"])]   # drop the 4 "XNA" rows

test["AGE_GROUP"] = pd.cut(test["AGE_YEARS"], bins=[0, 30, 50, 100],
                           labels=["Under 30", "30-50", "Over 50"])

for group in ["CODE_GENDER", "AGE_GROUP"]:
    summary = test.groupby(group, observed=True).agg(
        clients=("TARGET", "size"),
        observed_default=("TARGET", "mean"),
        mean_pd=("PD", "mean"),
        mean_score=("SCORE", "mean"),
    )
    print(summary.style.format({"observed_default": "{:.2%}", "mean_pd": "{:.2%}",
                                "mean_score": "{:.0f}"}).to_string(), "\n")

 clients observed_default mean_pd mean_score
CODE_GENDER    
F 60826 7.01% 7.47% 570
M 31426 10.14% 9.24% 563
 

 clients observed_default mean_pd mean_score
AGE_GROUP    
Under 30 13559 11.29% 12.51% 551
30-50 47602 8.66% 8.20% 567
Over 50 31091 5.77% 5.94% 576
 



### The proxy test

In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import cross_val_score

X = woe_transform(test, binners)[FINAL_VARS]
is_female = (test["CODE_GENDER"] == "F").astype(int)

auc_all = cross_val_score(LogisticRegression(max_iter=1000), X, is_female,
                          cv=5, scoring="roc_auc").mean()
print(f"All 12 variables predicting gender: AUC {auc_all:.3f}\n")

single = pd.Series({
    var: max(roc_auc_score(is_female, X[var]), 1 - roc_auc_score(is_female, X[var]))
    for var in FINAL_VARS
}).sort_values(ascending=False)
print(single.round(3))

All 12 variables predicting gender: AUC 0.720

OCCUPATION_TYPE                0.675
EXT_SOURCE_1                   0.605
ORGANIZATION_TYPE              0.604
DAYS_EMPLOYED                  0.564
OWN_CAR_AGE                    0.535
EXT_SOURCE_3                   0.518
REGION_RATING_CLIENT_W_CITY    0.511
CREDIT_TERM                    0.510
AMT_GOODS_PRICE                0.509
NAME_EDUCATION_TYPE            0.508
EXT_SOURCE_2                   0.508
CREDIT_GOODS_RATIO             0.500
dtype: float64


### Disparate impact, the 80% rule

In [3]:
cutoff = test["SCORE"].quantile(0.20)
test["APPROVED"] = test["SCORE"] >= cutoff
print(f"Score cut-off: {cutoff:.0f}  (overall approval {test['APPROVED'].mean():.1%})\n")

for group in ["CODE_GENDER", "AGE_GROUP"]:
    rates = test.groupby(group, observed=True)["APPROVED"].mean()
    di = (rates / rates.max()).rename("ratio_vs_highest")
    table = pd.concat([rates.rename("approval_rate"), di], axis=1)
    table["passes_80_rule"] = table["ratio_vs_highest"] >= 0.8
    print(table.style.format({"approval_rate": "{:.1%}",
                              "ratio_vs_highest": "{:.2f}"}).to_string(), "\n")

Score cut-off: 545  (overall approval 80.0%)

 approval_rate ratio_vs_highest passes_80_rule
CODE_GENDER   
F 82.7% 1.00 True
M 74.7% 0.90 True
 

 approval_rate ratio_vs_highest passes_80_rule
AGE_GROUP   
Under 30 59.8% 0.67 False
30-50 79.4% 0.89 True
Over 50 89.7% 1.00 True
 



### Calibration by group

In [4]:
test["PD_BAND"] = pd.cut(test["PD"], bins=[0, 0.03, 0.06, 0.10, 0.20, 1.0],
                         labels=["<3%", "3-6%", "6-10%", "10-20%", ">20%"])

for group in ["CODE_GENDER", "AGE_GROUP"]:
    calib = test.groupby([group, "PD_BAND"], observed=True).agg(
        clients=("TARGET", "size"), predicted=("PD", "mean"), observed=("TARGET", "mean"),
    )
    calib["gap_pp"] = (calib["observed"] - calib["predicted"]) * 100
    print(calib.style.format({"predicted": "{:.2%}", "observed": "{:.2%}",
                              "gap_pp": "{:+.2f}"}).to_string(), "\n")

  clients predicted observed gap_pp
CODE_GENDER PD_BAND    
F <3% 15982 1.93% 1.95% +0.02
3-6% 18040 4.37% 3.77% -0.60
6-10% 12668 7.78% 7.34% -0.44
10-20% 10507 13.74% 13.11% -0.64
>20% 3629 28.00% 26.56% -1.44
M <3% 5962 1.97% 2.30% +0.33
3-6% 8337 4.40% 4.91% +0.50
6-10% 6910 7.83% 9.19% +1.36
10-20% 7126 13.94% 14.96% +1.02
>20% 3091 28.62% 30.38% +1.76
 

  clients predicted observed gap_pp
AGE_GROUP PD_BAND    
Under 30 <3% 1148 2.10% 2.00% -0.10
3-6% 2488 4.53% 3.78% -0.75
6-10% 3228 7.94% 7.43% -0.51
10-20% 4297 14.21% 13.54% -0.66
>20% 2398 28.87% 24.69% -4.19
30-50 <3% 10672 1.93% 1.99% +0.05
3-6% 13710 4.40% 4.27% -0.13
6-10% 10238 7.80% 8.47% +0.67
10-20% 9487 13.79% 14.58% +0.79
>20% 3495 28.25% 30.82% +2.57
Over 50 <3% 10124 1.92% 2.10% +0.18
3-6% 10179 4.32% 4.03% -0.29
6-10% 6112 7.71% 7.49% -0.22
10-20% 3849 13.47% 12.42% -1.06
>20% 827 26.75% 28.30% +1.54
 



## Findings

**Proxies:** although gender is excluded, the 12 model variables predict gender with an AUC of 0.720. The strongest proxies are OCCUPATION_TYPE (0.675), EXT_SOURCE_1 (0.605) and ORGANIZATION_TYPE (0.604). EXT_SOURCE_1 is an external score of undisclosed origin, which raises a transparency issue.

**Disparate impact (80% rule, cut-off approving 80% of applicants):** gender passes (ratio 0.90 for men). Clients under 30 fail (ratio 0.67): 59.8% are approved versus 89.7% of clients over 50.

**Calibration by group:** the model is systematically miscalibrated by group. At the same predicted PD, women default less often than predicted and men more often, so women are effectively held to a stricter standard. Clients under 30 are overestimated in every PD band, most strongly in the riskiest band (predicted 28.9%, observed 24.7%), which contributes to their low approval rate.

**Interpretation:** part of the group differences reflects real differences in default rates. However, the systematic over-prediction for women and young clients is not justified by their actual behaviour. Correcting gender miscalibration would require using gender, which anti-discrimination rules prohibit; this is a known trade-off, since different fairness definitions cannot all be satisfied when groups have different default rates.

## Limitations

- The 80% rule is a screening heuristic from US employment guidelines, not an EU legal standard; failing it calls for investigation rather than establishing discrimination.
- Only gender and age are tested; other protected characteristics are not available in the data.
- Whether a difference is legally justified is a legal and compliance question, beyond the scope of this analysis.
- In a bank, mitigation options (alternative variables, additional data for thin-file applicants, recalibration) would be assessed and decided through model governance.